# M6.0d Primer: From Maxwell's Equations to the Yee Grid

Book pointer: chapter 17, section 17.2 (Maxwell's equations). The numerical method in this notebook is not in the book; it is built here from the two curl equations, in the same way that primer M2.0b built the relaxation and sparse solvers from Poisson's equation.

Fourth primer for module 6. Primer 00c derived electromagnetic waves from Maxwell's equations and gave the closed forms: the wave speed $c = 1/\sqrt{\mu_0\epsilon_0}$, the impedance $\eta_0 = \sqrt{\mu_0/\epsilon_0}$, reflection and transmission at a dielectric step, the Poynting vector. This notebook turns the same equations into an algorithm, the finite-difference time-domain (FDTD) method of Yee, which is what `emlib.fdtd` implements and what notebooks 03 and 04 use. Everything is done first on a grid small enough to update with a pencil, then on real grids, and every property of the method (stability, dispersion, boundaries, sources, energy) is measured as well as derived.

## 1. Warm-up

Try these on paper first. Answers are in section 7.

1. Write the two curl equations of Maxwell's equations in vacuum. Which one says how $\mathbf{B}$ changes in time, and what sets the rate?
2. For a plane wave moving along $x$ with $\mathbf{E}$ along $y$, which component of $\mathbf{B}$ is non-zero, and how are $E_y$ and $B$ related in size?
3. The central difference $(f(x + h) - f(x - h))/2h$ is second-order accurate. Why is the forward difference $(f(x + h) - f(x))/h$ only first order, and at which point is it secretly second order?
4. A wave moves at speed $c$. If the grid spacing is $\Delta x$ and the time step $\Delta t$, how many cells does the wave cross in one step? What do you expect to go wrong if that number exceeds one?
5. On a grid, a wave with 4 cells per wavelength and one with 40 cells per wavelength are both supposed to move at $c$. Which do you trust more, and why?
6. What boundary condition does a perfect conductor impose on the tangential electric field, and what happens to a wave that hits it?
7. How much energy per unit area does a plane wave of amplitude $E_0$ carry past a point per second?

## 2. The ideas

**Why step Maxwell's equations directly.** Primer 00c combined Faraday's law and the Ampere-Maxwell law into a single second-order wave equation for $\mathbf{E}$. That is the right move on paper, because the wave equation has known solutions. On a computer it is better to keep the two first-order equations as they are. They are an update rule already: Faraday's law says the rate of change of $\mathbf{B}$ at a point is set by the curl of $\mathbf{E}$ around it, and the Ampere-Maxwell law says the rate of change of $\mathbf{E}$ is set by the curl of $\mathbf{H}$ (minus any current). Know $\mathbf{E}$ and $\mathbf{H}$ now, and you know how both change in the next instant. Step forward, repeat. Materials enter through $\epsilon$ and $\sigma$ at each point, boundaries through what you do at the edge, sources through a term you add. The method needs no matrix and no iteration to convergence: each step is a handful of array subtractions.

**Staggering.** The obvious way to discretise a derivative, a difference between neighbouring values divided by the spacing, is only first-order accurate unless the difference is centred on the point where the derivative is wanted. Yee's idea (1966) is to arrange the grid so that every derivative is centred. The magnetic field is stored half a cell away from the electric field, so the difference of two neighbouring $E$ values is centred exactly on an $H$ value, and the other way round. In time the same trick is played: $H$ is stored half a time step out of phase with $E$, so the change of $H$ over one step is centred on the moment at which $E$ is known. The fields leapfrog over each other. The result is a scheme that is second-order accurate in both space and time, uses only nearest neighbours, and is remarkably faithful to the structure of Maxwell's equations: the discrete divergence of $\mathbf{B}$ stays zero for ever if it starts at zero, and a discrete energy is conserved exactly in a lossless region.

**Stability.** An explicit time-stepping scheme has a speed limit. In one step, information on the grid can move only from a node to its neighbours, one cell. A wave moves $c\,\Delta t$ in one step. If $c\,\Delta t > \Delta x$, the true solution at a node depends on values the scheme has not looked at, and the scheme cannot represent it. The penalty is not a small error but an exponential blow-up: the shortest wavelength the grid can hold grows by a fixed factor every step. The dimensionless ratio $S = c\,\Delta t/\Delta x$ is the Courant number, and the Yee scheme is stable for $S \le 1$ in one dimension, $S \le 1/\sqrt2$ in two and $S \le 1/\sqrt3$ in three. Section 3.3 derives these limits by inserting a single Fourier mode into the update.

**Dispersion.** Even a stable scheme does not move every wavelength at exactly $c$. A wave sampled with only a few points per wavelength is seen by the difference formulas as having a slightly different curvature than it really has, and it travels a little too slowly. Short wavelengths lag, long ones are nearly exact, so a sharp pulse slowly spreads into a ripple trailing behind it. This numerical dispersion is the main accuracy limit of FDTD, and it sets the practical rule used in notebooks 03 and 04: at least 20 cells per wavelength in the slowest medium, for every wavelength the source contains. In one dimension at exactly $S = 1$ the dispersion vanishes and the scheme is exact, which makes it the best possible test case.

**Boundaries and sources.** A grid is finite, so something must be done at its edge. A perfect electric conductor (PEC) is easy, $E = 0$ on the boundary, and reflects everything. To mimic open space the boundary must instead absorb what reaches it. In 1D a one-line rule derived from the one-way wave equation (Mur's boundary) does this almost perfectly; in 2D, `emlib.fdtd.fdtd_2d_tm` surrounds the domain with a frame of graded lossy material whose electric and magnetic conductivities are matched so that a wave enters it without reflecting and dies inside. Waves are started either by an initial field, which gives full control over direction, or by adding a source value at a node every step, which is how a current or an antenna drives a field.

**Units and what is measured.** `emlib.fdtd` works in SI units: $E$ in V/m, $H$ in A/m, $\Delta x$ in metres. The impedance $\eta_0 = 376.7\,\Omega$ converts between them, so $\eta_0 H$ has the same size as $E$ in a vacuum plane wave. Each run returns the fields, time series at chosen probe nodes (with $E$ and $H$ brought to the same place and time, so that the Poynting flux $E_yH_z$ can be formed), and snapshots. The energy $\tfrac12\epsilon E^2 + \tfrac12\mu_0H^2$ summed over the grid is the natural check that nothing has gone wrong.

**What the main notebooks do.** Notebook 03 runs the 1D solver: it measures the pulse speed, the Courant limit and the dispersion, the reflection and transmission at a dielectric step against primer 00c's formulas, attenuation in a conductor, and the energy carried by the Poynting flux. Notebook 04 runs the 2D solver: a point source sends out cylindrical waves, the absorbing frame lets them leave, and the Poynting vector $\mathbf{S}(x, y, t)$ shows energy flowing outward and through openings. Everything those notebooks rely on is derived and tested below.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import fdtd as fd
from emlib.fields import EPS0
from emlib.biot_savart import MU0

plt.rcParams["figure.figsize"] = (7, 4)
np.set_printoptions(precision=4, suppress=True, linewidth=110)
c, eta0 = fd.C0, fd.ETA0
print(f"c = {c:.6e} m/s,  eta0 = sqrt(mu0/eps0) = {eta0:.4f} ohm")

### 3.1 The one-dimensional equations

Start from the two curl equations in a linear medium with permittivity $\epsilon = \epsilon_r\epsilon_0$, permeability $\mu_0$ and conductivity $\sigma$:
$$\frac{\partial\mathbf{B}}{\partial t} = -\nabla\times\mathbf{E}, \qquad \epsilon\frac{\partial\mathbf{E}}{\partial t} = \nabla\times\mathbf{H} - \sigma\mathbf{E}, \qquad \mathbf{B} = \mu_0\mathbf{H}.$$
The second is the Ampere-Maxwell law with the conduction current $\mathbf{J} = \sigma\mathbf{E}$ written out. Now suppose nothing depends on $y$ or $z$ (a plane wave moving along $x$) and that $\mathbf{E}$ points along $y$. The curl of $\mathbf{E} = E_y(x, t)\,\hat{\mathbf{y}}$ has only a $z$ component, $\partial E_y/\partial x$, so Faraday's law gives a magnetic field along $z$ only, and the curl of $\mathbf{H} = H_z(x, t)\,\hat{\mathbf{z}}$ has only a $y$ component, $-\partial H_z/\partial x$. Two scalar equations remain:
$$\mu_0\frac{\partial H_z}{\partial t} = -\frac{\partial E_y}{\partial x}, \qquad \epsilon\frac{\partial E_y}{\partial t} = -\frac{\partial H_z}{\partial x} - \sigma E_y.$$
The Poynting vector $\mathbf{E}\times\mathbf{H} = E_yH_z\,(\hat{\mathbf{y}}\times\hat{\mathbf{z}}) = E_yH_z\,\hat{\mathbf{x}}$ points along $x$, as it must for a wave travelling along $x$. These are the equations `emlib.fdtd.fdtd_1d` solves.

Units. $E_y$ is in V/m and $H_z$ in A/m. Their ratio in a plane wave is an impedance, measured in ohms. In vacuum it is $\eta_0 = \sqrt{\mu_0/\epsilon_0} = 376.73\,\Omega$, and in a dielectric $\eta = \eta_0/\sqrt{\epsilon_r}$.

**Worked example 1 (a plane wave satisfies both equations).** Take $E_y = f(x - ct)$ for any smooth $f$, and $H_z = f(x - ct)/\eta_0$, in vacuum with $\sigma = 0$. Write $u = x - ct$. Then $\partial E_y/\partial x = f'(u)$ and $\partial H_z/\partial t = -c f'(u)/\eta_0$. The first equation requires $\mu_0(-cf'/\eta_0) = -f'$, that is $\mu_0c = \eta_0$. Check: $\mu_0c = \mu_0/\sqrt{\mu_0\epsilon_0} = \sqrt{\mu_0/\epsilon_0} = \eta_0$. The second requires $\epsilon_0(-cf') = -f'/\eta_0$, that is $\epsilon_0c\eta_0 = 1$, which is $\epsilon_0\sqrt{\mu_0/\epsilon_0}/\sqrt{\mu_0\epsilon_0} = 1$. Both hold for any shape $f$: the pulse keeps its shape and moves at $c$. A wave moving the other way, $f(x + ct)$, needs $H_z = -f/\eta_0$. The sign of $H$ relative to $E$ encodes the direction, through the sign of $E_yH_z$.

The cell checks both equations for a Gaussian pulse by finite differences, and the minus sign for the backward wave.

In [ ]:
f = lambda u: np.exp(-(u / 0.01) ** 2)                     # a 1 cm Gaussian
x0, t0, h, dt = 0.004, 2e-11, 1e-6, 1e-15
for sign, name in ((+1, "forward  f(x - ct), H = +E/eta0"), (-1, "backward f(x + ct), H = -E/eta0")):
    E = lambda x, t: f(x - sign * c * t)
    H = lambda x, t: sign * f(x - sign * c * t) / eta0
    dE_dx = (E(x0 + h, t0) - E(x0 - h, t0)) / (2 * h)
    dH_dx = (H(x0 + h, t0) - H(x0 - h, t0)) / (2 * h)
    dE_dt = (E(x0, t0 + dt) - E(x0, t0 - dt)) / (2 * dt)
    dH_dt = (H(x0, t0 + dt) - H(x0, t0 - dt)) / (2 * dt)
    print(f"{name}:  mu0 dH/dt + dE/dx = {MU0 * dH_dt + dE_dx:+.2e}   eps0 dE/dt + dH/dx = {EPS0 * dE_dt + dH_dx:+.2e}"
          f"   (size of each term: {abs(dE_dx):.1f} and {abs(dH_dx):.3f})")

### 3.2 Staggering in space and time: the leapfrog

Put $E_y$ on the nodes $x_i = i\,\Delta x$ and $H_z$ on the half nodes $x_{i+1/2} = (i + \tfrac12)\Delta x$. The derivative $\partial E_y/\partial x$ is needed in the $H$ equation, at the $H$ positions: the central difference $(E_{i+1} - E_i)/\Delta x$ is exactly centred there. The derivative $\partial H_z/\partial x$ is needed in the $E$ equation at the $E$ positions: $(H_{i+1/2} - H_{i-1/2})/\Delta x$ is centred there too. Every spatial derivative is a central difference over one cell, with error of order $\Delta x^2$.

Time is treated the same way. Store $E$ at integer times $t^n = n\,\Delta t$ and $H$ at half-integer times $t^{n+1/2}$. Then the change of $H$ from $t^{n-1/2}$ to $t^{n+1/2}$ is centred on $t^n$, when $E$ is known, and the change of $E$ from $t^n$ to $t^{n+1}$ is centred on $t^{n+1/2}$, when $H$ is known. In vacuum the update reads
$$H_{i+1/2}^{n+1/2} = H_{i+1/2}^{n-1/2} - \frac{\Delta t}{\mu_0\Delta x}\left(E_{i+1}^n - E_i^n\right), \qquad E_i^{n+1} = E_i^n - \frac{\Delta t}{\epsilon\,\Delta x}\left(H_{i+1/2}^{n+1/2} - H_{i-1/2}^{n+1/2}\right).$$
This is exactly the pair in the docstring of `emlib.fdtd`, which writes them as in-place array updates (`Hz -= ch * (Ey[1:] - Ey[:-1])`, then `Ey[1:-1] -= cb * (Hz[1:] - Hz[:-1])`) and adds the conductivity term described in section 3.5. The order matters: $H$ is updated first, from the current $E$; then $E$ from the new $H$. Each field jumps over the other, hence leapfrog. Note the two coefficients $c_h = \Delta t/(\mu_0\Delta x)$ and $c_e = \Delta t/(\epsilon\Delta x)$; with the Courant number $S = c\,\Delta t/\Delta x$ they are $c_h = S/\eta_0$ and, in vacuum, $c_e = S\eta_0$.

A clean way to see the update is to use $h = \eta_0H$ as the magnetic variable, which has the units and size of $E$. In vacuum the update becomes
$$h_{i+1/2} \leftarrow h_{i+1/2} - S\,(e_{i+1} - e_i), \qquad e_i \leftarrow e_i - S\,(h_{i+1/2} - h_{i-1/2}).$$
At $S = 1$ there is nothing left but subtractions.

**Worked example 2 (three steps by hand at $S = 1$).** Eight nodes, $x_0$ to $x_7$, with $e = (0, 0.5, 1, 0.5, 0, 0, 0, 0)$ at $t = 0$. To make the pulse move right only, set $h = e$ shifted to match a forward wave: for $E = f(x - ct)$ the magnetic field at the half node $x_{i+1/2}$ and time $-\Delta t/2$ is $f(x_{i+1/2} + c\Delta t/2)/\eta_0 = f(x_{i+1})/\eta_0$ at $S = 1$. So $h_{i+1/2} = e_{i+1}$: $h = (0.5, 1, 0.5, 0, 0, 0, 0)$ on the seven half nodes.

Step 1, $h$ first, $h_{i+1/2} \leftarrow h_{i+1/2} - (e_{i+1} - e_i)$:
$h_{1/2} = 0.5 - (0.5 - 0) = 0$; $h_{3/2} = 1 - (1 - 0.5) = 0.5$; $h_{5/2} = 0.5 - (0.5 - 1) = 1$; $h_{7/2} = 0 - (0 - 0.5) = 0.5$; the rest stay 0. So $h = (0, 0.5, 1, 0.5, 0, 0, 0)$.

Then $e$ at the interior nodes, $e_i \leftarrow e_i - (h_{i+1/2} - h_{i-1/2})$:
$e_1 = 0.5 - (0.5 - 0) = 0$; $e_2 = 1 - (1 - 0.5) = 0.5$; $e_3 = 0.5 - (0.5 - 1) = 1$; $e_4 = 0 - (0 - 0.5) = 0.5$; $e_5 = e_6 = 0$. So $e = (0, 0, 0.5, 1, 0.5, 0, 0, 0)$.

The whole pulse, $e$ and $h$ together, has moved one cell to the right, with no change of shape. Steps 2 and 3 do the same: after three steps $e = (0, 0, 0, 0, 0.5, 1, 0.5, 0)$. At $S = 1$ the 1D Yee scheme moves a wave exactly one cell per step, which is exactly what a wave moving at $c$ does in a time $\Delta x/c$. This is the "magic time step", and section 3.3 shows why it is also the largest stable one.

The cell reproduces the hand calculation with `fd.fdtd_1d`, passing the initial fields as `Ey0` and `Hz0` (the latter is interpreted at $t = -\Delta t/2$, as in the hand example). The boundary nodes are held at zero, which does not matter for three steps.

In [ ]:
dx = 1e-3
e0 = np.array([0, 0.5, 1, 0.5, 0, 0, 0, 0.0])
h0 = e0[1:].copy()                                          # h_{i+1/2} = e_{i+1}: a forward wave at S = 1
print("t = 0:      e =", e0, "   eta0 H =", h0)
for steps in (1, 2, 3):
    r = fd.fdtd_1d(7, dx, steps, courant=1.0, Ey0=e0, Hz0=h0 / eta0, boundary="pec")
    print(f"step {steps}:     e =", np.round(r["Ey"], 12) + 0.0, "   eta0 H =", np.round(r["Hz"] * eta0, 12) + 0.0)
print(f"time step at S = 1 and dx = 1 mm: dt = {r['dt']:.4e} s = dx / c")

**Animation.** Two rows of bars: $E$ on the nodes and $H$ on the half nodes between them. The update alternates, first the $H$ row (each bar changes by the difference of the two $E$ bars beside it), then the $E$ row, and each row is highlighted as it changes, so the leapfrog is visible as a rhythm. A Gaussian pulse moves right. In the second part the Courant number is set to 1.02, just above the limit of section 3.3, and a readout of the largest $|E|$ shows the solution growing without bound after some tens of steps.

In [ ]:
from emlib import show
show.animation("m6_waves", "YeeLeapfrog")

### 3.3 Stability and the Courant number

The Courant number $S = c\,\Delta t/\Delta x$ is the distance a wave travels in one step, measured in cells. Worked example 2 ran at $S = 1$. What happens for other values?

**Von Neumann analysis.** In a uniform vacuum grid with no boundaries, any field can be written as a sum of Fourier modes, and the update treats each mode separately. Insert one mode, $e_i^n = \hat e\,\mathrm{e}^{\mathrm{i}(k x_i - \omega t^n)}$ and $h_{i+1/2}^{n+1/2} = \hat h\,\mathrm{e}^{\mathrm{i}(k x_{i+1/2} - \omega t^{n+1/2})}$, with a wavenumber $k$ and an angular frequency $\omega$ that may be complex. A difference over one cell of $\mathrm{e}^{\mathrm{i}kx}$ gives $\mathrm{e}^{\mathrm{i}kx}(\mathrm{e}^{\mathrm{i}k\Delta x/2} - \mathrm{e}^{-\mathrm{i}k\Delta x/2}) = 2\mathrm{i}\sin(k\Delta x/2)\,\mathrm{e}^{\mathrm{i}kx}$, and a difference over one step of $\mathrm{e}^{-\mathrm{i}\omega t}$ gives $-2\mathrm{i}\sin(\omega\Delta t/2)\,\mathrm{e}^{-\mathrm{i}\omega t}$. The two update equations become
$$-2\mathrm{i}\sin(\omega\Delta t/2)\,\hat h = -S\cdot2\mathrm{i}\sin(k\Delta x/2)\,\hat e, \qquad -2\mathrm{i}\sin(\omega\Delta t/2)\,\hat e = -S\cdot2\mathrm{i}\sin(k\Delta x/2)\,\hat h.$$
Multiply them: $\sin^2(\omega\Delta t/2) = S^2\sin^2(k\Delta x/2)$, so
$$\sin\!\left(\frac{\omega\Delta t}{2}\right) = S\sin\!\left(\frac{k\Delta x}{2}\right).$$
This is the numerical dispersion relation of the 1D Yee scheme. (The exact relation is $\omega = ck$; for small $k\Delta x$ and $\omega\Delta t$ the sines become their arguments and it is recovered.)

Stability: a mode stays bounded if $\omega$ is real, because then $\mathrm{e}^{-\mathrm{i}\omega t}$ only rotates. $\omega$ is real exactly when the right-hand side lies in $[-1, 1]$ for every $k$. The largest value of $\sin(k\Delta x/2)$ is 1, at $k\Delta x = \pi$, the shortest wave the grid can hold (two cells per wavelength, a pattern $+, -, +, -$). So the condition is $S \le 1$. If $S > 1$, that mode has $\sin(\omega\Delta t/2) = S > 1$, which forces $\omega\Delta t/2 = \pi/2 + \mathrm{i}\gamma$ with $\cosh\gamma = S$, and the mode is multiplied by $\mathrm{e}^{2\gamma}$ in magnitude every step. Round-off error always contains a little of every mode, so it blows up.

In two dimensions the same analysis gives $\sin^2(\omega\Delta t/2) = S^2[\sin^2(k_x\Delta x/2) + \sin^2(k_y\Delta y/2)]$, and the worst mode (checkerboard, $k_x\Delta x = k_y\Delta y = \pi$) needs $2S^2 \le 1$: $S \le 1/\sqrt2$. In three dimensions $S \le 1/\sqrt3$. The reason in words: on a square grid the worst wave travels diagonally, and in one step it must not outrun the nearest-neighbour coupling along both axes at once.

**Worked example 3 (the largest time step).** For $\Delta x = 1\,$mm: in 1D $\Delta t \le \Delta x/c = 10^{-3}/2.998\times10^8 = 3.336\times10^{-12}\,$s. In 2D, divide by $\sqrt2$: $2.359\times10^{-12}\,$s. In 3D, divide by $\sqrt3$: $1.926\times10^{-12}\,$s. Halving the cell size halves the time step too, so a 2D simulation of a fixed physical time costs $2^3 = 8$ times more when the grid is refined by 2, and a 3D one 16 times more.

**Worked example 4 (how fast it blows up).** At $S = 1.02$, $\cosh\gamma = 1.02$ gives $\gamma = 0.1997$ and a growth factor of $\mathrm{e}^{2\gamma} = 1.491$ per step. A smooth starting pulse contains the checkerboard mode only at round-off level, about $10^{-16}$. Growing by 1.491 per step it reaches order 1 after $\ln(10^{16})/0.399 \approx 92$ steps. The code runs the update itself in the variables $e$ and $h = \eta_0H$ (because `emlib.fdtd` refuses $S > 1$) for $S = 0.5$, $1.0$ and $1.02$, and measures the growth rate.

In [ ]:
def leapfrog_run(S, steps=200, n=400):
    e = np.exp(-((np.arange(n + 1) - 200) / 10.0) ** 2)    # smooth pulse, both ends held at zero
    h = np.zeros(n)
    peak = []
    for _ in range(steps):
        h -= S * (e[1:] - e[:-1])
        e[1:-1] -= S * (h[1:] - h[:-1])
        peak.append(np.abs(e).max())
    return np.array(peak), e

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for S in (0.5, 1.0, 1.02):
    peak, e_end = leapfrog_run(S)
    ax[0].semilogy(peak, label=f"S = {S}")
    if S == 1.02:
        rate = (peak[199] / peak[150]) ** (1 / 49)
        first = np.argmax(peak > 10)
        ax[1].plot(e_end[180:220] / np.abs(e_end).max(), ".-", color="C2")
ax[0].set_xlabel("step"); ax[0].set_ylabel("max |E|"); ax[0].legend(); ax[0].set_title("bounded for S <= 1, exponential growth above")
ax[1].set_title("S = 1.02 after 200 steps: the +,-,+,- mode"); ax[1].set_xlabel("node (zoom)")
plt.tight_layout(); plt.show()
gamma = np.arccosh(1.02)
print(f"S = 1.02: measured growth per step {rate:.4f}, von Neumann exp(2 arccosh S) = {np.exp(2 * gamma):.4f}; max |E| first exceeds 10 at step {first}")
for d, name in ((1, "1D"), (2, "2D"), (3, "3D")):
    print(f"{name}: largest stable dt for dx = 1 mm is {fd.courant_limit(d) * 1e-3 / c:.4e} s")

### 3.4 Numerical dispersion

For $S \le 1$ the dispersion relation of section 3.3 has a real $\omega$ for every $k$, but $\omega$ is not exactly $ck$. Write the number of cells per wavelength as $N = 2\pi/(k\Delta x)$. Solving for $\omega$ and dividing by $k$ gives the numerical phase velocity
$$\frac{v_p}{c} = \frac{\omega}{ck} = \frac{2\arcsin\!\big(S\sin(\pi/N)\big)}{S\cdot2\pi/N}.$$
Two things follow at once. At $S = 1$, $\arcsin(\sin(\pi/N)) = \pi/N$ and $v_p = c$ for every $N$: no dispersion at all in 1D. For $S < 1$ the velocity is below $c$ and falls as $N$ decreases. Expanding for large $N$,
$$1 - \frac{v_p}{c} \approx \frac{\pi^2(1 - S^2)}{6N^2},$$
so the error falls as the square of the resolution (second order, as promised by the central differences) and is largest for small $S$.

**Worked example 5 (10 and 20 points per wavelength at $S = 0.5$).** For $N = 10$: $\sin(\pi/10) = 0.30902$, times $S$ gives 0.15451, $\arcsin$ gives 0.15513, doubled 0.31026, divided by $S\cdot2\pi/10 = 0.31416$: $v_p/c = 0.98759$, a 1.24 percent lag. The approximation gives $\pi^2\cdot0.75/600 = 1.23$ percent. For $N = 20$: $v_p/c = 0.99691$, a 0.31 percent lag, four times smaller. A 0.3 percent lag sounds small, but it is a phase error: after the wave has travelled 100 wavelengths it is behind by $2\pi\cdot100\cdot0.0031 = 1.9\,$rad, a third of a cycle. That is why accurate long-distance propagation needs fine grids, and why the working rule is at least 20 cells per wavelength, in the medium where the wavelength is shortest (in a dielectric with $\epsilon_r = 4$ the wavelength is half its vacuum value).

The code plots $v_p/c$ against $N$ for several $S$, then measures it. A pulse made of a sine carrier with a Gaussian envelope is launched at a frequency whose vacuum wavelength is 10 cells, and recorded at two probes 500 cells apart. Because each frequency travels independently through a linear grid, the phase of the Fourier component at the carrier frequency changes between the probes by exactly $k(\omega)\times500\,\Delta x$, so the numerical wavenumber can be read off and compared with the dispersion relation solved for $k$: $\sin(k\Delta x/2) = \sin(\omega\Delta t/2)/S$.

In [ ]:
vp = lambda N, S: 2 * np.arcsin(S * np.sin(np.pi / N)) / (S * 2 * np.pi / N)
N = np.linspace(3, 40, 300)
for S in (0.25, 0.5, 0.9, 1.0):
    plt.plot(N, vp(N, S), label=f"S = {S}")
plt.axvline(20, color="grey", ls=":"); plt.xlabel("cells per wavelength N"); plt.ylabel("v_p / c"); plt.ylim(0.85, 1.01)
plt.legend(); plt.title("numerical phase velocity of the 1D Yee scheme"); plt.show()
print(f"S = 0.5: v_p/c = {vp(10, 0.5):.5f} at N = 10, {vp(20, 0.5):.5f} at N = 20;  S = 1: {vp(10, 1.0):.5f}")

# measurement: carrier with vacuum wavelength 10 cells, S = 0.5
S, dx = 0.5, 1e-3
dt = S * dx / c
w0 = 2 * np.pi * c / (10 * dx)
tw = 40 * dt; tc = 5 * tw
packet = lambda t: np.exp(-((t - tc) / tw) ** 2) * np.sin(w0 * (t - tc))
run = fd.fdtd_1d(3000, dx, 9000, courant=S, source=packet, src_index=500, probes=[1000, 1500])
t = run["t"]
P = [np.sum(run["probe_E"][:, k] * np.exp(-1j * w0 * t)) for k in range(2)]
k_th = (2 / dx) * np.arcsin(np.sin(w0 * dt / 2) / S)
dphi = np.mod(np.angle(P[0] / P[1]), 2 * np.pi)
k_meas = (dphi + 2 * np.pi * np.round((k_th * 500 * dx - dphi) / (2 * np.pi))) / (500 * dx)
print(f"carrier with vacuum wavelength 10 cells: numerical wavelength {2 * np.pi / (k_th * dx):.3f} cells, v_p/c = {w0 / (c * k_th):.5f}")
print(f"measured k / dispersion-relation k = {k_meas / k_th:.6f}   (vacuum k / dispersion-relation k = {(w0 / c) / k_th:.5f})")

### 3.5 Materials: permittivity and conductivity on the grid

A dielectric enters through $\epsilon = \epsilon_r\epsilon_0$ in the $E$ update, one value per $E$ node: `eps_r` in `fdtd_1d` is an array on the nodes. The wave speed inside is $c/\sqrt{\epsilon_r}$, so the local Courant number is $S/\sqrt{\epsilon_r}$, smaller than in vacuum; stability is set by the fastest region (usually vacuum), and accuracy by the slowest (where the wavelength is shortest). The permeability is $\mu_0$ everywhere in this course.

Where $\epsilon_r$ jumps between two nodes, the discrete interface sits somewhere between them, and the reflection it produces has the right size but a phase that depends on that position at the level of a cell. This is the same issue as the half-node interface of the dielectric solver in module 3, and it matters only when you compare phases at the scale of $\Delta x$. Averaging $\epsilon_r$ on the node where the jump occurs is the standard fix when it matters.

Conductivity is the more interesting case. The $E$ equation is now $\epsilon\,\partial E/\partial t = -\partial H/\partial x - \sigma E$, and the loss term $\sigma E$ must be evaluated at the half step $t^{n+1/2}$ to keep the scheme centred, where $E$ is not stored. The natural choice is the average, $\sigma(E^n + E^{n+1})/2$. The update becomes
$$\epsilon\frac{E^{n+1} - E^n}{\Delta t} = -\frac{H_{i+1/2} - H_{i-1/2}}{\Delta x} - \sigma\frac{E^{n+1} + E^n}{2},$$
which can be solved for $E^{n+1}$:
$$E^{n+1} = c_a E^n - c_b\left(H_{i+1/2} - H_{i-1/2}\right), \qquad c_a = \frac{1 - \sigma\Delta t/2\epsilon}{1 + \sigma\Delta t/2\epsilon}, \qquad c_b = \frac{\Delta t/(\epsilon\Delta x)}{1 + \sigma\Delta t/2\epsilon}.$$
These are the arrays `ca` and `cb` in `emlib.fdtd`. With $\sigma = 0$, $c_a = 1$ and $c_b = \Delta t/(\epsilon\Delta x)$, the vacuum update. The key property is $|c_a| < 1$ for every $\sigma > 0$, however large: the semi-implicit average never makes the loss term unstable. A fully explicit choice, $\sigma E^n$, gives $c_a = 1 - \sigma\Delta t/\epsilon$, which is below $-1$ (unstable) once $\sigma\Delta t/\epsilon > 2$.

**Worked example 6 (two conductors at $\Delta x = 1\,$mm, $S = 0.5$, so $\Delta t = 1.668\times10^{-12}\,$s).** Seawater, $\sigma = 4\,$S/m, $\epsilon_r = 81$: $\sigma\Delta t/2\epsilon = 4\times1.668\times10^{-12}/(2\times81\times8.854\times10^{-12}) = 4.65\times10^{-3}$, so $c_a = 0.99074$, a gentle loss of about 0.9 percent of $E$ per step. Copper, $\sigma = 5.96\times10^7\,$S/m, $\epsilon_r = 1$: $\sigma\Delta t/2\epsilon_0 = 5.6\times10^6$, so $c_a = -0.9999996$ and $c_b = 3.4\times10^{-5}\,\Omega$, essentially zero. The explicit choice would give $c_a = 1 - 2\times5.6\times10^6 = -1.1\times10^7$ and explode on the first step. With the semi-implicit coefficients the field in copper just flips sign each step while being fed almost nothing: in effect $E \approx 0$, a perfect conductor. Physically that is right. The charge in copper relaxes in $\epsilon_0/\sigma = 1.5\times10^{-19}\,$s, and at 10 GHz the skin depth is $\sqrt{2/\omega\mu_0\sigma} = 0.65\,\mu$m, a thousand times smaller than a cell. A metal on a millimetre grid is a PEC.

**A weakly lossy medium.** If $\sigma/(\omega\epsilon) \ll 1$ for the frequencies in a pulse, the pulse keeps its shape while its amplitude decays as $\mathrm{e}^{-\alpha x}$ with $\alpha = \sigma\eta/2 = \sigma\eta_0/(2\sqrt{\epsilon_r})$ (primer 00c). The code measures the peak at two probes 200 cells apart inside a lossy region and compares with $\mathrm{e}^{-\alpha\cdot0.2\,\mathrm{m}}$. The agreement improves as $\sigma/(\omega\epsilon_0)$, evaluated at the pulse's typical frequency $2/w$ for a Gaussian of width $w$, gets smaller: the formula is a low-loss limit, not a property of the grid.

In [ ]:
S, dx = 0.5, 1e-3
dt = S * dx / c
for name, sig, er in (("seawater", 4.0, 81.0), ("copper", 5.96e7, 1.0)):
    a = sig * dt / (2 * er * EPS0)
    ca, cb = (1 - a) / (1 + a), (dt / (er * EPS0 * dx)) / (1 + a)
    print(f"{name:9s} sigma dt / 2 eps = {a:.3e}   ca = {ca:+.7f}   cb = {cb:.3e} ohm   explicit 1 - sigma dt/eps = {1 - 2 * a:+.3e}")

# the two schemes on the bare loss equation eps dE/dt = -sigma E in copper, five steps
a = 5.96e7 * dt / (2 * EPS0)
E_semi, E_expl = [1.0], [1.0]
for _ in range(5):
    E_semi.append((1 - a) / (1 + a) * E_semi[-1]); E_expl.append((1 - 2 * a) * E_expl[-1])
print("semi-implicit:", np.array(E_semi)); print("explicit:     ", np.array(E_expl))

# attenuation in a weakly lossy region starting at node 1000
n = 3000
for sig0, wst in ((0.01, 40), (0.02, 20), (0.02, 40)):
    sig = np.zeros(n + 1); sig[1000:] = sig0
    w = wst * dt
    r = fd.fdtd_1d(n, dx, int(2800 / S), courant=S, sigma=sig, source=lambda t: fd.gaussian(t, 4 * w, w), src_index=500, probes=[1100, 1300])
    pk = r["probe_E"].max(axis=0)
    alpha = sig0 * eta0 / 2
    print(f"sigma = {sig0} S/m, width {wst} steps: sigma/(eps0 * 2/w) = {sig0 / (EPS0 * 2 / w):.3f};  peak ratio {pk[1] / pk[0]:.4f}, "
          f"exp(-alpha * 0.2 m) = {np.exp(-alpha * 0.2):.4f}, ratio {pk[1] / pk[0] / np.exp(-alpha * 0.2):.4f}")

### 3.6 Boundaries

**PEC.** Setting $E = 0$ on the end nodes is a perfect electric conductor: the tangential electric field at a conductor's surface is zero (primer 00c). A wave that arrives is reflected completely with its $E$ inverted, because the reflected wave must cancel the incident $E$ at the wall. The magnetic field is reflected without inversion, so $H$ doubles at the wall: that is the surface current a real conductor would carry.

**Mur's absorbing boundary.** To mimic an open end, the boundary node must take whatever value a wave leaving the grid would give it. At the left end only left-moving waves should exist, $E = f(x + ct)$, which satisfy the one-way wave equation
$$\frac{\partial E}{\partial t} - c\frac{\partial E}{\partial x} = 0.$$
Discretise it at the point midway between nodes 0 and 1 and midway between steps $n$ and $n + 1$, by averaging: the time derivative is $\tfrac12[(E_0^{n+1} - E_0^n) + (E_1^{n+1} - E_1^n)]/\Delta t$ and the space derivative $\tfrac12[(E_1^n - E_0^n) + (E_1^{n+1} - E_0^{n+1})]/\Delta x$. Solving for the one unknown, $E_0^{n+1}$:
$$E_0^{n+1} = E_1^n + \frac{S - 1}{S + 1}\left(E_1^{n+1} - E_0^n\right).$$
This is the line `Ey[0] = e1 + kl * (Ey[1] - e0)` in `fdtd_1d`, with $k = (S - 1)/(S + 1)$ computed from the local wave speed. Because it is centred, it is second-order accurate.

**Worked example 7 (Mur at $S = 1$ and $S = 0.5$).** At $S = 1$, $k = 0$ and the rule is $E_0^{n+1} = E_1^n$: the boundary takes the value its neighbour had one step earlier. Worked example 2 showed that at $S = 1$ waves move exactly one cell per step, so this is exactly what an outgoing wave would do, and the boundary absorbs perfectly, up to round-off. At $S = 0.5$, $k = -1/3$. Now the one-way equation is discretised with an error, and a small reflection remains. It shrinks as the pulse gets smoother, since the error is a high-order derivative of the pulse. The code measures it by launching a pulse towards the left boundary (with $E$ and $H$ matched for a left-moving wave, section 3.7) and recording what comes back past a probe.

**The 2D absorbing frame.** In two dimensions a wave hits the edge at every angle, and Mur's one-line rule is only exact at normal incidence. `fdtd_2d_tm` instead surrounds the domain with a frame `n_layer` cells deep in which the electric conductivity rises smoothly (as the cube of the depth) from zero to a maximum, and the magnetic field gets a matching magnetic conductivity $\sigma_m = \sigma\mu_0/\epsilon_0$. That choice makes the impedance of the lossy material, $\sqrt{(\mu_0 + \sigma_m/\mathrm{i}\omega)/(\epsilon_0 + \sigma/\mathrm{i}\omega)}$, equal to $\eta_0$ at every frequency, so a normally incident wave enters without any reflection and is absorbed inside. The grading keeps the jump from one cell to the next small, which matters because the discretised profile itself reflects a little where it changes. At oblique incidence the match is not perfect, and some reflection remains; a true perfectly matched layer (PML) removes it by making the loss direction-dependent, which this simple layer does not attempt. The code measures what is left in a 2D domain after a pulse from a point source has left: a fraction of a percent of the outgoing amplitude, against complete trapping with PEC walls.

In [ ]:
def boundary_reflection(width_cells, S, boundary="mur", n=800, dx=1e-3):
    # left-moving pulse launched at x = 0.4 m; the probe at node 200 sees it pass, then sees the reflection
    x = np.arange(n + 1) * dx; xh = (np.arange(n) + 0.5) * dx
    dt = S * dx / c
    f = lambda s: np.exp(-((s - 0.4) / (width_cells * dx)) ** 2)
    r = fd.fdtd_1d(n, dx, int(900 / S), courant=S, Ey0=f(x), Hz0=-f(xh - c * dt / 2) / eta0, probes=[200], boundary=boundary)
    travel = r["t"] * c / dx                                 # distance travelled, in cells
    p = r["probe_E"][:, 0]
    incident = p[travel < 350][np.argmax(np.abs(p[travel < 350]))]
    later = p[travel > 450]
    reflected = later[np.argmax(np.abs(later))]
    return reflected / incident

print("Mur boundary, |reflected / incident| for Gaussian pulses of half-width w cells:")
print("   w     S = 0.5     S = 0.9     S = 1.0")
for wc in (2, 5, 10, 20):
    print(f"  {wc:3d}  " + "  ".join(f"{abs(boundary_reflection(wc, S)):9.1e}" for S in (0.5, 0.9, 1.0)))
print(f"PEC wall, reflected / incident = {boundary_reflection(20, 0.5, 'pec'):+.4f}  (E inverted)")

# 2D: a point-source pulse leaves through the absorbing frame or is trapped by PEC walls
S2 = 0.5; dt2 = S2 * dx / c; w = 20 * dt2
src = lambda t: fd.gaussian(t, 4 * w, w)
early = fd.fdtd_2d_tm(301, 301, dx, 900, courant=S2, source=src, src_index=(150, 150), probes=[(190, 150)])
ref_amp = early["probe_E"][:, 0].max()
for n_layer, name in ((20, "graded lossy frame"), (0, "PEC walls")):
    late = fd.fdtd_2d_tm(301, 301, dx, 1400, courant=S2, source=src, src_index=(150, 150), n_layer=n_layer)
    print(f"2D, {name:18s}: largest |Ez| left in the domain = {np.abs(late['Ez']).max() / ref_amp:.2e} of the pulse amplitude 40 cells from the source")

### 3.7 Sources

A wave has to come from somewhere. Two kinds of source are used.

**Hard source.** Overwrite a node every step, $E_s^n = f(t^n)$. Simple, but the node is then a wall for any wave that comes back to it: whatever arrives is reflected, because the node's value is no longer free. Hard sources are rarely used for that reason.

**Soft source.** Add to the node every step, $E_s \leftarrow E_s + f(t)$, after the normal update. The node still obeys Maxwell's equations for whatever passes through it, and the added value acts like a current sheet: it is equivalent to a term $-J\Delta t/\epsilon$ in the Ampere-Maxwell update. This is what `fdtd_1d` does with `source` and `src_index`. A current sheet radiates equally in both directions, so in 1D a soft source launches two pulses, one each way. How big are they? Each step the source adds $f$ to one node; spread over the $1/S$ steps a wave takes to cross a cell, and shared between two directions, the radiated amplitude is $f/(2S)$. At $S = 0.5$ a Gaussian with peak 1 gives two pulses of amplitude 1.

**One-way launch.** To send a single pulse in one direction, start from fields that already form a travelling wave: $E_y = f(x)$ on the nodes and, for a wave moving to $+x$, $H_z = f(x)/\eta_0$ on the half nodes. Because $H$ is stored half a step earlier than $E$ (at $t = -\Delta t/2$), it must be the wave as it was then, shifted back: $H_z(x_{i+1/2}) = f(x_{i+1/2} + c\Delta t/2)/\eta_0$. For a wave moving to $-x$, $H_z = -f(x_{i+1/2} - c\Delta t/2)/\eta_0$. Worked example 2 used the first form at $S = 1$, where the shift is half a cell. At $S < 1$ the launch is matched only to the continuum wave, not to the slightly dispersive grid wave, so a tiny backward wave is also created, at the level of the dispersion error.

**Gaussian pulses and their spectrum.** A Gaussian in time, $f(t) = \exp(-(t - t_0)^2/w^2)$, has a Gaussian spectrum $\propto\exp(-\omega^2w^2/4)$. It contains all frequencies, but those above a few times $1/w$ are negligible. The spectrum falls to $10^{-3}$ of its peak at $\omega = 2\sqrt{\ln1000}/w = 5.26/w$, where the vacuum wavelength is $2\pi c/\omega = 1.19\,cw$. The pulse must also start smoothly, so $t_0$ is chosen as about $4w$ (the source then switches on from $\mathrm{e}^{-16} = 10^{-7}$).

**Worked example 8 (choosing a pulse width).** At $\Delta x = 1\,$mm and $S = 0.5$, a width of $w = 30\Delta t$ gives $cw = 30\times0.5\,\Delta x = 15\,\Delta x$, so the shortest significant wavelength is $1.19\times15 = 18$ cells. That is acceptable in vacuum, close to the 20-cell rule. In a dielectric with $\epsilon_r = 4$ the same frequencies have half the wavelength, 9 cells, and the pulse would visibly disperse; a width of $60\Delta t$ (36 cells in vacuum, 18 inside) is the right choice there. This is why the dielectric-step test in `tests/test_fdtd.py` uses $w = 60\Delta t$.

**The 2D point source.** In `fdtd_2d_tm` a soft source at one node is a line current along $z$ through that node, which radiates a cylindrical wave. Its amplitude falls as $1/\sqrt r$ (energy spread over a circumference $2\pi r$), and its waveform has a tail behind the front, a feature of 2D wave propagation. That tail shifts the peak of the received signal by a fixed amount, so the speed of a 2D wave is measured from the delay between two radii, not from a single arrival time.

The cell checks the soft-source amplitude $f/(2S)$ at three Courant numbers and the one-way launch.

In [ ]:
dx = 1e-3
for S in (0.25, 0.5, 1.0):
    dt = S * dx / c
    w = 20 * dx / c                                           # the same physical width at each S
    r = fd.fdtd_1d(2000, dx, int(2000 / S), courant=S, source=lambda t: fd.gaussian(t, 4 * w, w), src_index=1000, probes=[600, 1400])
    print(f"soft source, S = {S}: amplitude to the left {r['probe_E'][:, 0].max():.4f}, to the right {r['probe_E'][:, 1].max():.4f}, f/(2S) = {1 / (2 * S):.4f}")

S = 0.5; dt = S * dx / c; n = 1200
x = np.arange(n + 1) * dx; xh = (np.arange(n) + 0.5) * dx
f = lambda s: np.exp(-((s - 0.3) / 0.03) ** 2)
r = fd.fdtd_1d(n, dx, 1200, courant=S, Ey0=f(x), Hz0=f(xh + c * dt / 2) / eta0, snapshot_every=400)
plt.plot(x, f(x), "k:", label="t = 0")
for k, snap in enumerate(r["snapshots"]):
    plt.plot(x, snap, label=f"after {400 * (k + 1)} steps")
plt.xlabel("x [m]"); plt.ylabel("Ey [V/m]"); plt.legend(); plt.title("a one-way launch at S = 0.5"); plt.show()
left = np.abs(r["snapshots"][-1][x < 0.3]).max()
print(f"after 1200 steps the pulse has moved {(x[np.argmax(r['snapshots'][-1])] - 0.3) / dx:.0f} cells (expected {1200 * S:.0f}); "
      f"largest |E| left behind: {left:.1e}")

### 3.8 Measuring things: probes, Poynting flux and energy

**Probes.** A probe records a field at one node every step, which is how speeds, reflection coefficients and spectra are measured. For the energy flow both $E$ and $H$ are needed at the same place and time, which the staggered grid never provides directly. `fdtd_1d` brings them together for each probe: $H$ is averaged over the two half nodes beside the probe node (to get the place right) and over the two half steps either side of the current time (to get the time right). Both averages are second-order accurate. The Poynting flux at the probe is then $S_x = E_yH_z$ in W/m$^2$, and its time integral is the energy per unit area that passed.

**Energy on the grid.** The natural discrete energy per unit area is $U = \sum\tfrac12\epsilon E_i^2\Delta x + \sum\tfrac12\mu_0H_{i+1/2}^2\Delta x$, which `fd.energy_1d` computes. In a closed lossless cavity it should be constant, but with $E$ and $H$ at different times it is not exactly: it wobbles at the level of the time-staggering error. The quantity the leapfrog conserves exactly is the mixed form
$$W^n = \sum\tfrac12\epsilon\left(E_i^n\right)^2\Delta x + \sum\tfrac12\mu_0H_{i+1/2}^{n-1/2}H_{i+1/2}^{n+1/2}\Delta x,$$
with the magnetic term built from the two half steps either side of $t^n$. (Multiply the $E$ update by $E^n + E^{n+1}$ and the $H$ update by the neighbouring $H$ values, sum over the grid, and the cross terms telescope.) The wobble in $U$ is therefore not a loss of energy, only a mismatch of times, and it does not grow.

**Worked example 9 (energy of a pulse and its flux).** A one-way pulse with $E_y = f$ and $\eta_0H_z = f$ has equal electric and magnetic energy densities, $\tfrac12\epsilon_0f^2$ and $\tfrac12\mu_0f^2/\eta_0^2 = \tfrac12\epsilon_0f^2$ (since $\mu_0/\eta_0^2 = \epsilon_0$). Its energy per unit area is $\epsilon_0\int f^2dx$. For $f = \mathrm{e}^{-(x/a)^2}$ with $a = 3\,$cm, $\int f^2dx = a\sqrt{\pi/2} = 0.0376\,$m, so $U = 8.854\times10^{-12}\times0.0376 = 3.33\times10^{-13}\,$J/m$^2$ for a peak field of 1 V/m. As the pulse passes a probe, $\int E_yH_z\,dt = \int f^2/\eta_0\,dt = \int f^2dx/(c\eta_0) = \epsilon_0\int f^2dx$: the same number. The code checks this, and the energy behaviour in a PEC cavity.

In [ ]:
S, dx = 0.5, 1e-3
dt = S * dx / c
n = 1200; x = np.arange(n + 1) * dx; xh = (np.arange(n) + 0.5) * dx
f = lambda s: np.exp(-((s - 0.3) / 0.03) ** 2)
Ey0, Hz0 = f(x), f(xh + c * dt / 2) / eta0
U0 = fd.energy_1d(Ey0, Hz0, dx)
r = fd.fdtd_1d(n, dx, 4000, courant=S, Ey0=Ey0, Hz0=Hz0, probes=[800])
flux = np.sum(r["probe_E"][:, 0] * r["probe_H"][:, 0]) * dt
print(f"pulse energy on the grid {U0:.4e} J/m^2, hand value eps0 a sqrt(pi/2) = {EPS0 * 0.03 * np.sqrt(np.pi / 2):.4e};  "
      f"integral of E H at the probe {flux:.4e} (ratio {flux / U0:.4f})")

# energy in a closed PEC cavity: U wobbles, the mixed form W is constant
n = 400; x = np.arange(n + 1) * dx
e = np.exp(-((x - 0.2) / 0.02) ** 2); e[0] = e[-1] = 0
h = np.zeros(n)
ch, ce = dt / (MU0 * dx), dt / (EPS0 * dx)
U, W = [], []
for k in range(3000):
    h_old = h.copy()
    h -= ch * (e[1:] - e[:-1])
    U.append(fd.energy_1d(e, h, dx))
    W.append(0.5 * dx * (EPS0 * np.sum(e**2) + MU0 * np.sum(h_old * h)))
    e[1:-1] -= ce * (h[1:] - h[:-1])
U, W = np.array(U[10:]), np.array(W[10:])
plt.plot(U / W.mean(), label="U: E and H at different times"); plt.plot(W / W.mean(), label="W: the leapfrog invariant")
plt.xlabel("step"); plt.ylabel("energy / mean"); plt.legend(); plt.title("energy in a closed PEC cavity, S = 0.5"); plt.show()
print(f"peak-to-peak / mean: U {np.ptp(U) / U.mean():.2e},  W {np.ptp(W) / W.mean():.2e}")

### Exercise: one Yee step

Replace the body of `my_yee_step` with your own version of one interior update: $H$ on all half nodes from the current $E$, then $E$ on the interior nodes from the new $H$, with the coefficient arrays `ca`, `cb` on the nodes and the scalar `ch`, as in section 3.5. Leave the two end nodes alone (the check uses PEC ends, which stay zero). The check runs five of your steps on a pulse in a medium with a dielectric region and a lossy region, and compares with `fd.fdtd_1d`.

In [ ]:
def my_yee_step(Ey, Hz, ca, cb, ch):
    '''One leapfrog step in place: Hz on the half nodes, then Ey on the interior nodes.'''
    Hz -= ch * (Ey[1:] - Ey[:-1])                                # replace from here with your own version
    Ey[1:-1] = ca[1:-1] * Ey[1:-1] - cb[1:-1] * (Hz[1:] - Hz[:-1])
    return Ey, Hz


n, dx, S = 200, 1e-3, 0.5
dt = S * dx / c
x = np.arange(n + 1) * dx
eps_r = np.where(x > 0.12, 4.0, 1.0)
sigma = np.where((x > 0.15) & (x < 0.17), 0.5, 0.0)
eps = eps_r * EPS0
a = sigma * dt / (2 * eps)
ca, cb, ch = (1 - a) / (1 + a), (dt / (eps * dx)) / (1 + a), dt / (MU0 * dx)
Ey = np.exp(-((x - 0.1) / 0.01) ** 2); Ey[0] = Ey[-1] = 0.0
Hz = np.zeros(n)
ref = fd.fdtd_1d(n, dx, 5, courant=S, eps_r=eps_r, sigma=sigma, Ey0=Ey, Hz0=Hz, boundary="pec")
for _ in range(5):
    Ey, Hz = my_yee_step(Ey, Hz, ca, cb, ch)
print(f"after 5 steps: max |Ey - library| = {np.abs(Ey - ref['Ey']).max():.1e},  max |Hz - library| = {np.abs(Hz - ref['Hz']).max():.1e}")
assert np.allclose(Ey, ref["Ey"], atol=1e-12) and np.allclose(Hz, ref["Hz"], atol=1e-15)
print("exercise passed")

## 4. Verify

Three properties derived above, checked with asserts.

1. At $S = 1$ the scheme is exact in 1D: a one-way pulse moves exactly one cell per step. After 300 steps the field must equal the starting field shifted by 300 nodes, to round-off.
2. The numerical wavenumber measured in section 3.4 at a vacuum wavelength of 10 cells matches the dispersion relation $\sin(k\Delta x/2) = \sin(\omega\Delta t/2)/S$ to better than one part in a thousand, while differing from the vacuum value $\omega/c$ by more than one percent.
3. Mur's boundary at $S = 1$ reflects nothing beyond round-off: below $10^{-10}$.

In [ ]:
dx, n = 1e-3, 800
x = np.arange(n + 1) * dx; xh = (np.arange(n) + 0.5) * dx
f = lambda s: np.exp(-((s - 0.1) / 0.01) ** 2)
r = fd.fdtd_1d(n, dx, 300, courant=1.0, Ey0=f(x), Hz0=f(xh + dx / 2) / eta0)
shift_error = np.abs(r["Ey"][300:] - f(x)[:-300]).max()
print(f"1. S = 1, after 300 steps: max |E - E0 shifted by 300 nodes| = {shift_error:.1e}")
assert shift_error < 1e-12

print(f"2. measured k / dispersion-relation k = {k_meas / k_th:.6f};  vacuum k / dispersion-relation k = {(w0 / c) / k_th:.5f}")
assert abs(k_meas / k_th - 1) < 1e-3 and abs((w0 / c) / k_th - 1) > 1e-2

R1 = abs(boundary_reflection(10, 1.0))
print(f"3. Mur at S = 1: |reflected / incident| = {R1:.1e}")
assert R1 < 1e-10

## 5. Explore and rebuild

Explore. Numerical dispersion is anisotropic in 2D: a wave travelling along a grid axis lags more than one travelling along a diagonal. Derive the 2D phase velocity from $\sin^2(\omega\Delta t/2) = S^2[\sin^2(k_x\Delta x/2) + \sin^2(k_y\Delta x/2)]$ for a wave at angle $\theta$ to the $x$ axis, plot $v_p/c$ against $\theta$ at 10 cells per wavelength and $S = 0.5$, and then measure it: run `fd.fdtd_2d_tm` with a point source emitting a modulated Gaussian and compare the arrival of the carrier phase along the axis and along the diagonal. At which angle is the scheme most accurate, and at which Courant number does the anisotropy vanish (if it does)?

Rebuild. Close this notebook. On paper: derive the 1D update pair from the two curl equations, do worked example 2 again, insert a Fourier mode and derive $\sin(\omega\Delta t/2) = S\sin(k\Delta x/2)$ and the stability limit, derive $c_a$ and $c_b$ from the averaged loss term, and derive Mur's boundary from the one-way wave equation. In a blank editor: write a complete 1D Yee loop with a soft source and Mur boundaries, and reproduce the reflection table of section 3.6 and the speed and dispersion measurements of section 3.4.

## 6. Key results

| result | formula | holds when |
|---|---|---|
| 1D equations ($E$ along $y$, $H$ along $z$) | $\mu_0\partial_tH_z = -\partial_xE_y$, $\ \epsilon\partial_tE_y = -\partial_xH_z - \sigma E_y$ | plane wave along $x$ |
| impedance | $\eta_0 = \sqrt{\mu_0/\epsilon_0} = 376.73\,\Omega$; forward wave $H_z = E_y/\eta$, backward $H_z = -E_y/\eta$ | vacuum; $\eta = \eta_0/\sqrt{\epsilon_r}$ in a dielectric |
| Yee grid | $E$ on nodes at integer steps, $H$ on half nodes at half steps | |
| update | $H \leftarrow H - \frac{\Delta t}{\mu_0\Delta x}\Delta E$, then $E \leftarrow c_aE - c_b\Delta H$ | |
| Courant number | $S = c\,\Delta t/\Delta x$ | |
| stability | $S \le 1$ (1D), $1/\sqrt2$ (2D), $1/\sqrt3$ (3D) | uniform grid, explicit leapfrog |
| growth above the limit | factor $\mathrm{e}^{2\,\mathrm{arccosh}S}$ per step for the two-cell mode | 1D, $S > 1$ |
| numerical dispersion | $\sin(\omega\Delta t/2) = S\sin(k\Delta x/2)$ | 1D vacuum |
| phase velocity | $v_p/c = 2\arcsin(S\sin(\pi/N))/(2\pi S/N)$, $\ 1 - v_p/c \approx \pi^2(1 - S^2)/6N^2$ | $N$ cells per wavelength |
| magic time step | $S = 1$: exact, one cell per step | 1D vacuum only |
| resolution rule | at least 20 cells per wavelength in the slowest medium | for every frequency in the source |
| lossy coefficients | $c_a = \dfrac{1 - \sigma\Delta t/2\epsilon}{1 + \sigma\Delta t/2\epsilon}$, $\ c_b = \dfrac{\Delta t/\epsilon\Delta x}{1 + \sigma\Delta t/2\epsilon}$ | $\sigma E$ averaged over the step; $|c_a| < 1$ always |
| weak-loss attenuation | $\alpha = \sigma\eta/2$ | $\sigma/\omega\epsilon \ll 1$ |
| PEC | $E = 0$ at the wall; reflection $-1$ for $E$ | |
| Mur boundary | $E_0^{n+1} = E_1^n + \frac{S - 1}{S + 1}(E_1^{n+1} - E_0^n)$ | 1D; exact at $S = 1$ |
| matched lossy layer | $\sigma_m = \sigma\mu_0/\epsilon_0$ gives impedance $\eta_0$ | normal incidence exact; oblique approximate |
| soft source | $E_s \leftarrow E_s + f(t)$; radiates $f/(2S)$ each way | 1D |
| one-way launch | $E = f(x)$, $H = \pm f(x_{i+1/2} \pm c\Delta t/2)/\eta$ | $H$ stored at $t = -\Delta t/2$ |
| Gaussian spectrum | $\propto\mathrm{e}^{-\omega^2w^2/4}$; down to $10^{-3}$ at a wavelength of $1.19\,cw$ | |
| exact discrete energy | $W^n = \sum\tfrac12\epsilon(E^n)^2\Delta x + \sum\tfrac12\mu_0H^{n-1/2}H^{n+1/2}\Delta x$ | lossless, closed |
| pulse energy and flux | $U = \epsilon_0\int f^2dx = \int E_yH_z\,dt$ at a probe | one-way vacuum pulse |

## 7. Warm-up answers

1. $\partial\mathbf{B}/\partial t = -\nabla\times\mathbf{E}$ (Faraday) and $\epsilon_0\partial\mathbf{E}/\partial t = \nabla\times\mathbf{H}$ in vacuum (Ampere-Maxwell without current). The first says how $\mathbf{B}$ changes, at a rate set by the curl of $\mathbf{E}$.
2. $B_z$ (equivalently $H_z$). For a wave moving along $+x$, $E_y = cB_z$, or $E_y = \eta_0H_z$.
3. The forward difference has a leading error $\tfrac12hf''$, because the errors on the two sides do not cancel as they do in the central difference. It is a second-order estimate of the derivative at the midpoint $x + h/2$, which is exactly how the Yee grid uses it.
4. $S = c\,\Delta t/\Delta x$ cells. If $S > 1$ the wave would cross more than one cell per step while the scheme only couples neighbours, and the scheme becomes unstable: the shortest wavelength grows exponentially (section 3.3).
5. The 40-cell wave. With 4 cells per wavelength the difference formulas misjudge the curvature badly and the wave travels noticeably slower than $c$; the phase-velocity error falls as $1/N^2$ (section 3.4).
6. The tangential $\mathbf{E}$ vanishes at a perfect conductor. The wave is totally reflected, with $E$ inverted and $H$ doubled at the wall.
7. The intensity is $E_0H_0 = E_0^2/\eta_0$ at the peak of the wave; for a sinusoid the average is $E_0^2/2\eta_0$ W/m$^2$. That is the Poynting flux $E_yH_z$.